# Day 12 · Exercise 4: persistent_collection

**What you'll build:** `load_or_create_collection(path: str, name: str) -> chromadb.Collection` — a factory function that opens (or creates) a disk-backed Chroma collection with an `OllamaEmbeddingFunction` attached, so callers pass raw text and never touch a vector.

**Why it matters:** Wrapping the three-line setup — `PersistentClient`, `OllamaEmbeddingFunction`, `get_or_create_collection` — in a single factory is the pattern that keeps embedding wiring out of your application logic and lets every script in a project open the same store safely.

## Your Implementation

In [ ]:
import chromadb
import ollama


class OllamaEmbeddingFunction(chromadb.EmbeddingFunction):
    """Wraps Ollama's embeddings endpoint as a Chroma EmbeddingFunction."""

    def __init__(self, model: str = "nomic-embed-text") -> None:
        self.model = model

    def __call__(self, input: list[str]) -> list[list[float]]:
        vectors = []
        for text in input:
            response = ollama.embeddings(model=self.model, prompt=text)
            vectors.append(list(response["embedding"]))
        return vectors


def load_or_create_collection(path: str, name: str) -> chromadb.Collection:
    """Open an existing persistent Chroma collection or create it if absent.

    Creates a PersistentClient at *path*, attaches an OllamaEmbeddingFunction
    (model="nomic-embed-text"), and calls get_or_create_collection so the
    returned collection accepts plain text on both add() and query().

    Args:
        path: Directory for the Chroma SQLite store (created if it does not
              exist). Example: "./my_chroma_store".
        name: Name of the collection inside that store. Example: "docs".

    Returns:
        A chromadb.Collection with an OllamaEmbeddingFunction attached.
        Callers can add(documents=[...]) and query(query_texts=[...]) without
        passing any embeddings argument.

    Example:
        col = load_or_create_collection("./store", "notes")
        col.add(ids=["d0"], documents=["Chroma stores vectors on disk."])
        result = col.query(query_texts=["vector database"], n_results=1)
        print(result["documents"][0][0])
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
import tempfile
import os

_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(load_or_create_collection), 'load_or_create_collection is not defined'
        print(f'{_PASS} Check 1/{total}: load_or_create_collection is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: returns a chromadb.Collection
    _col = None
    _tmp_dir = tempfile.mkdtemp(prefix="chroma_ex04_")
    try:
        _col = load_or_create_collection(_tmp_dir, "test_collection")
        assert isinstance(_col, chromadb.Collection), (
            f'expected chromadb.Collection, got {type(_col).__name__}'
        )
        print(f'{_PASS} Check 2/{total}: returns a chromadb.Collection')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return

    # Check 3: add() accepts plain documents (no embeddings= argument)
    try:
        docs = [
            "Chroma persists vectors in SQLite files.",
            "EmbeddingFunction lets you add raw text without manual embedding.",
            "PersistentClient writes data to a directory on disk.",
        ]
        _col.add(ids=[f"doc_{i}" for i in range(len(docs))], documents=docs)
        count = _col.count()
        assert count == len(docs), f'expected {len(docs)} documents, got {count}'
        print(f'{_PASS} Check 3/{total}: add(documents=[...]) stores {count} documents without embeddings= argument')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: query() returns the most relevant document as a string
    try:
        results = _col.query(query_texts=["how does chroma store data on disk?"], n_results=1)
        best = results["documents"][0][0]
        assert isinstance(best, str), f'expected str, got {type(best).__name__}'
        assert "SQLite" in best or "disk" in best, (
            f'top result not semantically relevant: {best!r}'
        )
        print(f'{_PASS} Check 4/{total}: query returns the relevant document as a string')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Call `load_or_create_collection` twice with the **same path and name**. Add one document in the first call. Then query the collection returned by the second call — without adding anything — and verify the document is still there.

This foreshadows Day 12's project pattern: the ingestion script and the query script are separate processes that share one store. The factory function makes that seamless.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def load_or_create_collection(path: str, name: str) -> chromadb.Collection:
    ef = OllamaEmbeddingFunction(model="nomic-embed-text")
    client = chromadb.PersistentClient(path=path)
    return client.get_or_create_collection(name=name, embedding_function=ef)
```

**Why this works:** `PersistentClient` opens (or creates) the SQLite store at `path`, and `get_or_create_collection` is idempotent — it returns the existing collection if the name is already there, or creates a fresh one if not. Attaching `OllamaEmbeddingFunction` here means every `add(documents=[...])` and `query(query_texts=[...])` call routes through Ollama automatically, so application code never needs to manage vectors directly. The embedding function must be re-attached each session because Chroma stores vectors and text on disk but not the Python callable that produced them.
</details>